# Módulo 2: Regresión de Conteo Diario de Siniestros

## 1. Contexto y Objetivo
Este notebook evalúa si las características del calendario (mes, día de la semana, fin de semana, festivo) permiten predecir el volumen total diario de accidentes viales en la ciudad.

## 2. Enfoque Metodológico
- Agregación temporal a nivel de día (`fecha`).
- Evaluación de **Regresión Lineal** y **SGDRegressor** contra un **Baseline (Media histórica del conjunto de entrenamiento)**.
- Muestreo fuera de muestra utilizando el año **2022** completo como test.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv('C:\\Users\\santa\\Desktop\\Cositas de Python\\UNI\\IA\\proyectoIA\\data\\clean\\siniestralidad_clean.csv')

C:\Users\santa\AppData\Local\Temp\ipykernel_16432\2175947294.py:1: DtypeWarning: Columns (0: codigo, 1: medio_reporte, 2: direccion_original, 3: direccion_norm) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:\\Users\\santa\\Desktop\\Cositas de Python\\UNI\\IA\\proyectoIA\\data\\clean\\siniestralidad_clean.csv')


1. Agregación Diaria y Transformaciones

In [4]:
# Agrupar el dataset original por fecha
df_diario = (
    df.groupby('fecha')
    .agg(
        total_siniestros=('codigo', 'count'),
        anio=('anio', 'first'),
        mes=('mes', 'first'),
        dia_semana=('dia_semana', 'first'),
        es_fin_semana=('es_fin_semana', 'first'),
        es_festivo=('es_festivo', 'first'),
    )
    .reset_index()
)

# Encoding temporal
df_diario['es_fin_semana'] = df_diario['es_fin_semana'].astype(int)
df_diario['es_festivo'] = df_diario['es_festivo'].astype(int)
df_diario['mes_sin'] = np.sin(2 * np.pi * df_diario['mes'] / 12)
df_diario['mes_cos'] = np.cos(2 * np.pi * df_diario['mes'] / 12)
df_diario = pd.get_dummies(
    df_diario,
    columns=['dia_semana'],
    prefix='dia',
    drop_first=True,
    dtype=int,
)

feature_cols = [
    'anio',
    'mes_sin',
    'mes_cos',
    'es_fin_semana',
    'es_festivo',
] + [c for c in df_diario.columns if c.startswith('dia_')]

X = df_diario[feature_cols]
y = df_diario['total_siniestros']

df_diario.head()

,fecha,total_siniestros,anio,mes,es_fin_semana,es_festivo,mes_sin,mes_cos,dia_jueves,dia_lunes,dia_martes,dia_miércoles,dia_sábado,dia_viernes
0,2016-01-01,38,2016,1,0,1,0.5,0.866025,0,0,0,0,0,1
1,2016-01-02,25,2016,1,1,0,0.5,0.866025,0,0,0,0,1,0
2,2016-01-03,31,2016,1,1,0,0.5,0.866025,0,0,0,0,0,0
3,2016-01-04,30,2016,1,0,0,0.5,0.866025,0,1,0,0,0,0
4,2016-01-05,24,2016,1,0,0,0.5,0.866025,0,0,1,0,0,0


2. Split Temporal y Escalado

In [5]:
mask_train = df_diario['anio'] < 2022
mask_test = df_diario['anio'] == 2022

X_tr_raw, y_tr = X[mask_train], y[mask_train].values
X_te_raw, y_te = X[mask_test], y[mask_test].values

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)

3. Entrenamiento y Evaluación Comparativa

In [6]:
# 1. Baseline: Media constante del Train
y_baseline = np.full_like(y_te, fill_value=y_tr.mean(), dtype=float)

# 2. Mínimos Cuadrados Ordinarios (OLS)
ols = LinearRegression()
ols.fit(X_tr, y_tr)
y_pred_ols = ols.predict(X_te)

# 3. SGDRegressor
sgd = SGDRegressor(
    max_iter=2000, tol=1e-3, penalty=None, eta0=0.01, random_state=42
)
sgd.fit(X_tr, y_tr)
y_pred_sgd = sgd.predict(X_te)


def calcular_metricas(y_real, y_pred, nombre):
  return {
      'Modelo': nombre,
      'MAE (Siniestros/día)': mean_absolute_error(y_real, y_pred),
      'RMSE': np.sqrt(mean_squared_error(y_real, y_pred)),
      'R² Score': r2_score(y_real, y_pred),
  }


df_res = pd.DataFrame([
    calcular_metricas(y_te, y_baseline, 'Baseline (Media Train)'),
    calcular_metricas(y_te, y_pred_ols, 'Regresión Lineal (OLS)'),
    calcular_metricas(y_te, y_pred_sgd, 'SGDRegressor (Scikit-Learn)'),
])

display(df_res)

,Modelo,MAE (Siniestros/día),RMSE,R² Score
0,Baseline (Media Train),21.087660,22.974205,-5.321879
1,Regresión Lineal (OLS),13.581949,17.018906,-2.469193
2,SGDRegressor (Scikit-Learn),13.765809,17.201916,-2.544205


## Interpretación de Resultados: Ruptura Estructural y Diagnóstico del $R^2$ Negativo

### 1. Diagnóstico del Cambio de Nivel (Level Shift)
El coeficiente de determinación $R^2 < 0$ obtenido en el conjunto de prueba (2022) refleja una **ruptura estructural** entre el periodo de entrenamiento y el de evaluación:
- **Periodo de Entrenamiento (2016–2021):** Presenta un promedio diario histórico significativamente más alto ($\approx 40\text{--}50$ siniestros/día) debido al alto volumen de registros de "Solo daños" en los primeros años del dataset.
- **Periodo de Prueba (2022):** Registra un promedio diario de $\approx 27$ siniestros/día, producto del colapso en el reporte administrativo de accidentes sin lesionados post-pandemia.

Al evaluar 2022 con un *Baseline* calculado sobre el promedio del pasado, la constante sobreestima sistemáticamente la realidad en $\approx 21$ siniestros diarios (MAE = 21.08).

### 2. Comportamiento de OLS y SGDRegressor
Tanto la **Regresión Lineal (OLS)** como el **SGDRegressor** logran reducir sustancialmente el Error Absoluto Medio (el MAE baja de **21.08 a 13.58** siniestros/día) al aprovechar la pendiente negativa de la variable `anio`. Sin embargo, al tratarse de una serie inestacionaria con un salto discreto de nivel, la varianza del error cuadrático supera a la varianza respecto a la media del propio año 2022, resultando en $R^2 < 0$.

### 3. Decision
Este resultado negativo documentado es un hallazgo clave del proyecto:
> **Conclusión:** La serie de volumen diario de siniestros es inestacionaria y altamente estocástica; las variables de calendario (día, mes, festivos) no poseen poder predictivo estable a largo plazo. Este comportamiento justifica abandonar la planificación basada en fechas y enfocar la estrategia en la **inercia territorial constante del Top 200 de esquinas críticas (Módulo 3)**.

In [7]:
df['fecha'] = pd.to_datetime(df['fecha'])
df['grave'] = df['severidad'].isin(['Con lesionado', 'Con fallecido']).astype(int)

# Serie diaria de graves, con los días sin registros como 0
d = df.groupby('fecha').agg(graves=('grave', 'sum'), es_festivo=('es_festivo', 'max'))
d = d.reindex(pd.date_range(d.index.min(), d.index.max(), freq='D'))
d['graves'] = d['graves'].fillna(0)
d['es_festivo'] = d['es_festivo'].fillna(0).astype(int)

# Variables de calendario (sin 'anio')
d['mes_sin'] = np.sin(2 * np.pi * d.index.month / 12)
d['mes_cos'] = np.cos(2 * np.pi * d.index.month / 12)
d['fin_semana'] = (d.index.dayofweek >= 5).astype(int)
dow = pd.get_dummies(d.index.dayofweek, prefix='dia', drop_first=True, dtype=int)
dow.index = d.index
X = pd.concat([d[['mes_sin', 'mes_cos', 'fin_semana', 'es_festivo']], dow], axis=1)
y = d['graves']

# Régimen estable: train 2020-2022, test 2023
tr = (d.index.year >= 2020) & (d.index.year <= 2022)
te = d.index.year == 2023

sc = StandardScaler().fit(X[tr])
ols = LinearRegression().fit(sc.transform(X[tr]), y[tr])
sgd = SGDRegressor(max_iter=2000, tol=1e-3, penalty=None, eta0=0.01,
                   random_state=42).fit(sc.transform(X[tr]), y[tr])

pred = {
    'Media del train': np.full(te.sum(), y[tr].mean()),
    'Media móvil 28 días': y.shift(1).rolling(28).mean()[te].values,
    'OLS (calendario)': ols.predict(sc.transform(X[te])),
    'SGD (calendario)': sgd.predict(sc.transform(X[te])),
}
res = pd.DataFrame([{'Modelo': k,
                     'MAE': mean_absolute_error(y[te], v),
                     'RMSE': np.sqrt(mean_squared_error(y[te], v)),
                     'R²': r2_score(y[te], v)} for k, v in pred.items()]).round(3)
display(res)

,Modelo,MAE,RMSE,R²
0,Media del train,2.950,3.718,-0.019
1,Media móvil 28 días,2.913,3.725,-0.023
2,OLS (calendario),3.154,4.001,-0.180
3,SGD (calendario),3.160,4.012,-0.187


## Interpretación de Resultados

### 1. El volumen total diario no permite evaluar el calendario (cambio de nivel)
El promedio diario pasa de aprox 48 siniestros/día (2016-2021) a aprox 27 (2022) y casi 13 (2023). 

La caída se concentra en los "Solo daños" (17,142 en 2019 -> 5,464 en 2022 -> 566 en 2023), mientras los graves se mantienen en aprox 4,500 al año desde 2020. 

La causa no se puede establecer con estos datos, lo más probable es un cambio en qué eventos se registran, pero es una hipótesis.

Con ese salto, el error del baseline (MAE 21.1) es casi todo de nivel, y el R² negativo (−5.3) mide ese salto, no la utilidad del calendario. 

La mejora de OLS/SGD (MAE 13.6) viene de que `anio` extrapola una recta descendente, no de una habilidad del calendario, y aun así sobreestiman.

### 2. Prueba limpia: graves por día (train 2020-2022, test 2023, sin `anio`)

| Modelo | MAE | RMSE | R² |
|---|---|---|---|
| Media del train | 2.950 | 3.718 | -0.019 |
| Media móvil 28 días | 2.913 | 3.725 | -0.023 |
| OLS (calendario) | 3.154 | 4.001 | -0.180 |
| SGD (calendario) | 3.160 | 4.012 | -0.187 |

- Los modelos de calendario quedan peor que la media del entrenamiento (MAE aprox +7%).
- La media móvil de 28 días tampoco mejora: el nivel de graves es estable.
- OLS y SGD convergen a la misma solución (MAE 3.154 vs 3.160).
- Con llegadas puramente aleatorias (Poisson, media ≈ 12) el error típico sería ≈ 3.5, el mejor predictor obtiene 3.7. A escala diaria queda muy poco por explicar.

### 3. Relación con los efectos significativos del calendario
En el análisis exploratorio, mes y festivo mostraron diferencias estadísticamente significativas (festivo ≈ 20% menos graves). Esos efectos existen pero son pequeños frente al ruido diario; una explicación plausible es que estimar varios coeficientes con ese ruido empeora la predicción fuera de muestra.

### 4. Conclusión
Con las variables de calendario disponibles no se predice el volumen diario de siniestros graves mejor que con su media (≈ 12 por día en 2020-2023). El calendario, por sí solo, no justifica ajustes de dotación. El análisis territorial (módulo 3) se evalúa con su propia evidencia.

### Limitaciones
- Un único año de test (2023).
- 2020 (confinamiento) está en el entrenamiento y puede distorsionar los patrones de calendario.
- No se incluyeron otras variables (clima, eventos, obras).